# Пошаговый прогноз остатков с ARIMA и SARIMAX

На входе `df` с `REP_DT` и `OST_VAL`. Модель строит непрерывный прогноз на каждый день или месяц с одной даты выпуска. На исторических блоках факт нужен для оценки. После проверки строим прогноз от последнего известного остатка.

Если пакетов ещё нет, выполни `%pip install statsmodels "mlflow>=3,<4" "pandas>=2.2" "matplotlib>=3.7"` в отдельной ячейке.

## 1. Данные и настройки

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.statsmodels
import numpy as np
import pandas as pd
import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

CURRENCY = None
HORIZONS = list(range(1, 13))
N_FOLDS = 3
TEST_START = None
ORDER = (1, 1, 0)
SEASONAL_ORDER = None
YEAR_END_DAYS = 1
FUTURE_COLS = []
STAT_COLS = []

In [ ]:
data = df.copy()
currency_col = next((col for col in ("CURENCY", "CURRENCY") if col in data), None)
if currency_col:
    data = data.dropna(subset=[currency_col])
    if CURRENCY is None and data[currency_col].nunique() > 1:
        raise ValueError("Укажи CURRENCY в ячейке настроек.")
    if CURRENCY is not None:
        data = data[data[currency_col] == CURRENCY].copy()

dates = pd.to_datetime(data["REP_DT"], format="mixed", errors="coerce")
data["REP_DT"] = dates.dt.tz_localize(None).dt.normalize()
data = data.dropna(subset=["REP_DT"]).set_index("REP_DT").sort_index()
if data.empty or data.index.has_duplicates:
    raise ValueError("Нужен один ряд остатков с одной строкой на дату.")

monthly = data.index.to_period("M").is_unique
if monthly:
    data.index = data.index + pd.offsets.MonthEnd(0)
freq = pd.offsets.MonthEnd() if monthly else pd.offsets.Day()
data = data.asfreq(freq)
y = pd.to_numeric(data["OST_VAL"], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
last_fact = y.last_valid_index()
if last_fact is None:
    raise ValueError("В OST_VAL нет ни одного фактического остатка.")
if not HORIZONS or any(not isinstance(h, (int, np.integer)) or not 1 <= h <= 12 for h in HORIZONS):
    raise ValueError("Укажи целые горизонты от 1 до 12 месяцев.")
HORIZONS = sorted({int(h) for h in HORIZONS})
max_months = max(HORIZONS)
season = SEASONAL_ORDER or ((0, 0, 0, 0) if monthly else (1, 0, 0, 7))

def month_offset(months):
    return pd.offsets.MonthEnd(months) if monthly else pd.DateOffset(months=months)

future_end = last_fact + month_offset(max_months)
calendar_index = pd.date_range(data.index[0], future_end, freq=freq, name=data.index.name)
data = data.reindex(calendar_index)
y = y.reindex(calendar_index)

## 2. Признаки SARIMAX

Календарные признаки, включая конец года, строятся и после последнего факта. Если задаёшь свои `FUTURE_COLS`, добавь в `df` их значения до конца будущего горизонта в строках с пустым `OST_VAL`. Колонки `STAT_COLS` сдвигаются на 366 дней для дневного ряда или на 12 месяцев для месячного, чтобы прогноз мог использовать их в момент выпуска.

In [ ]:
exog = pd.DataFrame(index=calendar_index)
if monthly:
    angle = 2 * np.pi * exog.index.month / 12
    exog["calendar_month_sin"] = np.sin(angle)
    exog["calendar_month_cos"] = np.cos(angle)
else:
    exog["calendar_weekend"] = (exog.index.dayofweek >= 5).astype(float)
    exog["calendar_month_end"] = exog.index.is_month_end.astype(float)

if "is_year_end" not in FUTURE_COLS and "is_year_end" not in STAT_COLS:
    days_to_end = (exog.index + pd.offsets.YearEnd(0) - exog.index).days
    exog["is_year_end"] = (days_to_end < YEAR_END_DAYS).astype(float)
if "OST_VAL" in FUTURE_COLS or set(FUTURE_COLS) & set(STAT_COLS):
    raise ValueError("Укажи разные наборы признаков. Остаток можно добавить в STAT_COLS с лагом.")
future = data[FUTURE_COLS].apply(pd.to_numeric, errors="coerce").astype(float)
if not np.isfinite(future.to_numpy()).all():
    missing = future.index[~np.isfinite(future.to_numpy()).all(axis=1)][0]
    raise ValueError(f"Заполни FUTURE_COLS на календарной сетке. Первый пропуск {missing.date()}.")
stats = data[STAT_COLS].apply(pd.to_numeric, errors="coerce").astype(float)
stats = stats.replace([np.inf, -np.inf], np.nan).ffill().shift(12 if monthly else 366)
exog = exog.join(future).join(stats).dropna()
if exog.empty or last_fact not in exog.index:
    raise ValueError("Недостаточно истории для выбранных STAT_COLS.")
y = y.reindex(exog.index).asfreq(freq)
exog = exog.reindex(y.index)

## 3. Даты выпуска

У `train`, каждого фолда `valid` и `test` своя дата выпуска. Обучающая история для каждого прогноза заканчивается на его дате выпуска. Все валидационные пути заканчиваются к выпуску тестового прогноза. Валидационные пути могут перекрываться, так что качество сохраняется отдельно по фолдам.

Для `train` берём отдельный исторический блок до первой валидации. Для `test` по умолчанию берём последний доступный отрезок длиной до максимального горизонта. Если указан `TEST_START`, берём последний известный остаток строго перед этой датой.

In [ ]:
observed_dates = y.dropna().index
min_history = max(12 if monthly else 60, 2 * season[3])
origins = observed_dates[min_history - 1:]
ends = origins + month_offset(max_months)

if TEST_START is None:
    candidates = origins[ends <= last_fact]
else:
    cutoff = pd.Timestamp(TEST_START).tz_localize(None).normalize()
    candidates = origins[(origins < cutoff) & (origins < last_fact)]
if candidates.empty or N_FOLDS < 2:
    raise ValueError("Мало истории до теста. Проверь TEST_START, HORIZONS и N_FOLDS.")
test_origin = candidates[-1]
test_end = min(test_origin + month_offset(max_months), last_fact)

valid_candidates = origins[(origins >= ends[0]) & (ends <= test_origin)]
if len(valid_candidates) < N_FOLDS:
    raise ValueError("Мало истории для всех путей. Уменьши максимальный горизонт или число фолдов.")
valid_candidates = valid_candidates[min(len(valid_candidates) // 2, len(valid_candidates) - N_FOLDS):]
valid_origins = valid_candidates[np.linspace(0, len(valid_candidates) - 1, N_FOLDS, dtype=int)]
train_origin = origins[ends <= valid_origins[0]][-1]

blocks = [("train", None, train_origin, train_origin + month_offset(max_months))]
blocks += [
    ("valid", fold, origin, origin + month_offset(max_months))
    for fold, origin in enumerate(valid_origins, start=1)
]
blocks.append(("test", None, test_origin, test_end))
display(pd.DataFrame(blocks, columns=["split", "fold", "issue_date", "forecast_through"]))

## 4. Обучение и прогнозы

`get_forecast` строит всю траекторию из состояния на дату выпуска. Внутри пути не добавляем наблюдения и не меняем оценённые параметры. SARIMAX получает на будущих шагах только доступные заранее внешние признаки.

In [ ]:
def fit(name, origin):
    train = y.loc[:origin]
    center = train.mean()
    scale = train.std(ddof=0) or 1.0
    z = (train - center) / scale

    if name == "ARIMA":
        result = ARIMA(z, order=ORDER).fit()
    else:
        x = exog.loc[train.index]
        columns = x.loc[train.notna()].nunique().loc[lambda n: n > 1].index
        result = SARIMAX(
            z, exog=x[columns] if len(columns) else None,
            order=ORDER, seasonal_order=season,
        ).fit(disp=False)
    return result, center, scale


def forecast(fitted, origin, end):
    result, center, scale = fitted
    dates = pd.date_range(origin + freq, end, freq=freq)
    x_cols = None if isinstance(result.model, ARIMA) else result.model.exog_names
    x = exog.loc[dates, x_cols] if x_cols else None
    pred = result.get_forecast(steps=len(dates), exog=x)
    lower, upper = (pred.conf_int().to_numpy() * scale + center).T
    month_ends = pd.DatetimeIndex([origin + month_offset(h) for h in range(1, max_months + 1)])
    rows = pd.DataFrame({
        "REP_DT": origin, "target_dt": dates,
        "months": month_ends.searchsorted(dates) + 1,
        "target": y.reindex(dates).to_numpy(),
        "forecast": pred.predicted_mean.to_numpy() * scale + center,
        "baseline": y.loc[origin], "lower": lower, "upper": upper,
    })
    if not np.isfinite(rows["forecast"]).all():
        raise ValueError("Модель вернула некорректный прогноз. Проверь порядок и признаки.")
    return rows


def score(rows, by):
    error = (rows["target"] - rows["forecast"]).abs()
    baseline = (rows["target"] - rows["baseline"]).abs()
    denom = rows["target"].abs().replace(0, np.nan)
    return rows.assign(
        MAE=error, MAPE_pct=100 * error / denom,
        baseline_MAE=baseline, baseline_MAPE_pct=100 * baseline / denom,
    ).groupby(by, as_index=False).agg(
        MAE=("MAE", "mean"), MAPE_pct=("MAPE_pct", "mean"),
        baseline_MAE=("baseline_MAE", "mean"), baseline_MAPE_pct=("baseline_MAPE_pct", "mean"),
        n=("target", "count"), n_MAPE=("MAPE_pct", "count"),
    )

In [ ]:
if not mlflow.config.is_tracking_uri_set():
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("arima-sarimax-balances")
client = mlflow.MlflowClient()

models, model_uris, run_ids, target_scaling = {}, {}, {}, {}
predictions, future_predictions, metrics, cv_metrics = [], [], [], []

for name in ("ARIMA", "SARIMAX"):
    parts = []
    for split, fold, origin, end in blocks:
        fitted = fit(name, origin)
        parts.append(forecast(fitted, origin, end).assign(split=split, fold=fold))
        if split == "test":
            test_fitted = fitted

    rows = pd.concat(parts, ignore_index=True).assign(model=name)
    selected = rows[rows["months"].isin(HORIZONS)]
    scores = score(selected, ["model", "months", "split"])
    fold_scores = score(selected[selected["split"] == "valid"], ["model", "months", "fold"])
    predictions.append(rows)
    metrics.append(scores)
    cv_metrics.append(fold_scores)

    live_fitted = fit(name, last_fact)
    model, center, scale = live_fitted
    models[name] = model
    target_scaling[name] = (center, scale)
    live = forecast(live_fitted, last_fact, future_end).assign(model=name)
    future_predictions.append(live)

    with mlflow.start_run(run_name=name) as run:
        run_ids[name] = run.info.run_id
        mlflow.log_params({
            "order": ORDER, "seasonal_order": season if name == "SARIMAX" else None,
            "forecast_mode": "fixed_origin", "test_origin": str(test_origin.date()),
            "future_origin": str(last_fact.date()), "n_folds": N_FOLDS,
            "target_center": center, "target_scale": scale,
            "test_target_center": test_fitted[1], "test_target_scale": test_fitted[2],
            "exog": model.model.exog_names if name == "SARIMAX" else [],
            "converged": model.mle_retvals.get("converged"),
        })
        for row in scores.to_dict("records"):
            prefix = f"{row['split']}/{row['months']}m"
            mlflow.log_metrics({
                f"{prefix}/{key}": row[key] for key in ("MAE", "MAPE_pct") if pd.notna(row[key])
            })
        mlflow.log_text(scores.to_csv(index=False), "metrics.csv")
        mlflow.log_text(fold_scores.to_csv(index=False), "cv_metrics.csv")
        mlflow.log_text(live.to_csv(index=False), "future_forecast.csv")
        for suffix, saved in [("test", test_fitted[0]), ("future", model)]:
            info = mlflow.statsmodels.log_model(
                saved, name=f"{name.lower()}-{suffix}",
                pip_requirements=[f"statsmodels=={statsmodels.__version__}", f"pandas=={pd.__version__}"],
            )
            model_uris[name, suffix] = info.model_uri

predictions = pd.concat(predictions, ignore_index=True)
future_predictions = pd.concat(future_predictions, ignore_index=True)
metrics = pd.concat(metrics, ignore_index=True)
cv_metrics = pd.concat(cv_metrics, ignore_index=True)

## 5. Качество по выборкам и фолдам

Метрики горизонта `h` считаются по датам h-го месяца пути. Например, для горизонта 2 берём даты строго после отметки 1 месяц и до отметки 2 месяца включительно. Базовый прогноз остаётся равным остатку на дату выпуска. MAPE считается по ненулевым фактам.

Локальный MLflow открывается командой `mlflow ui --backend-store-uri sqlite:///mlflow.db`. При загрузке модели для будущего прогноза переведи результат в единицы остатка по формуле `raw_forecast * target_scale + target_center`. Для тестовой модели используй `test_target_scale` и `test_target_center`.

In [ ]:
display(metrics.sort_values(["months", "model", "split"]))
display(cv_metrics)

## 6. Факт и непрерывный прогноз

Каждая панель показывает путь от одной даты выпуска. Для `valid` можно выбрать фолд. Меняй `PLOT_MONTHS`, чтобы увеличить или сократить видимую часть пути. Метрики фолдов остаются в таблице выше.

In [ ]:
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter

PLOT_MONTHS = max_months
PLOT_VALID_FOLD = N_FOLDS
SHOW_TEST_INTERVAL = False

def draw_path(ax, part, show_interval=False):
    if show_interval:
        ax.fill_between(part["target_dt"], part["lower"], part["upper"],
                        color="#d45b26", alpha=0.13, label="95% интервал прогноза")
    ax.plot(part["target_dt"], part["target"], "-o" if monthly else "-",
            color="#185277", label="Факт")
    ax.plot(part["target_dt"], part["forecast"], "--s" if monthly else "--",
            color="#d45b26", label="Прогноз")
    locator = AutoDateLocator(minticks=3, maxticks=6)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(ConciseDateFormatter(locator))
    ax.grid(alpha=0.2)


for name in models:
    rows = predictions.query("model == @name and months <= @PLOT_MONTHS")
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), layout="constrained")
    fig.suptitle(f"{name} · первые {PLOT_MONTHS} мес.", fontsize=15)
    for split, ax in zip(("train", "valid", "test"), axes):
        part = rows[rows["split"] == split]
        label = split
        if split == "valid":
            part = part[part["fold"] == PLOT_VALID_FOLD]
            label = f"valid · фолд {PLOT_VALID_FOLD}"
        if part.empty:
            ax.set_title(f"{label} · нет прогноза", loc="left")
            continue

        ax.set_title(f"{label} · выпуск {part['REP_DT'].iloc[0]:%Y-%m-%d}", loc="left")
        draw_path(ax, part, split == "test" and SHOW_TEST_INTERVAL)
        ax.set_ylabel("OST_VAL")
        ax.legend(frameon=False)

    fig.supxlabel("Дата целевого остатка")
    plt.show()
    client.log_figure(run_ids[name], fig, f"fixed_path_{PLOT_MONTHS}m.png")

### Фиксированные горизонты на всём valid и test

На каждой панели показан один и тот же период целевых дат. Для факта на дату `t` прогноз на `h` месяцев выпускается в дату `t − h календарных месяцев`. Факт на конец марта для горизонта 1 месяц, например, сравнивается с прогнозом, выпущенным в конце февраля.

Для этой проверки `valid` делится на непересекающиеся отрезки. Его последние факты уже доступны к первому тестовому выпуску. В каждом фолде и на тесте параметры оцениваются к самому раннему выпуску. После этого модель обновляет состояние по фактам, известным на дату очередного выпуска. Каждый отдельный прогноз развивается пошагово от этой даты.

Таблицы ниже показывают MAE и MAPE по всем доступным фактам на каждом фиксированном горизонте. Прогноз после последней фактической даты расположен в следующем разделе.

In [ ]:
test_dates = pd.date_range(test_origin + freq, test_end, freq=freq)
first_test_issue = test_dates[0] - month_offset(max_months)
valid_dates = pd.date_range(valid_origins[0] + freq, first_test_issue, freq=freq)
if len(valid_dates) < N_FOLDS:
    raise ValueError("До первого тестового выпуска мало дат для valid. Уменьши максимальный горизонт.")

rolling_blocks = [
    ("valid", fold, pd.DatetimeIndex(dates))
    for fold, dates in enumerate(np.array_split(valid_dates.to_numpy(), N_FOLDS), start=1)
]
rolling_blocks.append(("test", None, test_dates))


def fixed_lead(name, target_dates):
    requests = pd.concat([
        pd.DataFrame({"target_dt": target_dates, "REP_DT": target_dates - month_offset(h), "months": h})
        for h in HORIZONS
    ], ignore_index=True).sort_values("REP_DT")
    first_issue = requests["REP_DT"].iloc[0]
    if y.loc[:first_issue].count() < 2:
        raise ValueError("До первого выпуска мало фактических остатков для обучения.")

    state, center, scale = fit(name, first_issue)
    seen = y.index.get_loc(first_issue) + 1
    scaled = (y - center) / scale
    x_cols = None if name == "ARIMA" else state.model.exog_names
    x = exog[x_cols] if x_cols else None
    baseline = y.ffill()
    parts = []
    for issue, group in requests.groupby("REP_DT", sort=False):
        start = y.index.get_loc(issue) + 1
        if seen < start:
            history_x = x.iloc[seen:start] if x is not None else None
            state = state.extend(scaled.iloc[seen:start], exog=history_x)
            seen = start
        positions = y.index.get_indexer(group["target_dt"])
        stop = positions.max() + 1
        future_x = x.iloc[start:stop] if x is not None else None
        path = state.get_forecast(steps=int(stop - start), exog=future_x)
        offsets = positions - start
        lower, upper = (path.conf_int().to_numpy()[offsets] * scale + center).T
        parts.append(group.assign(
            target=y.iloc[positions].to_numpy(), baseline=baseline.loc[issue],
            forecast=path.predicted_mean.to_numpy()[offsets] * scale + center,
            lower=lower, upper=upper, fit_through=first_issue,
        ))
    result = pd.concat(parts, ignore_index=True)
    if not np.isfinite(result["forecast"]).all():
        raise ValueError(f"{name} вернула некорректный прогноз с фиксированным упреждением.")
    return result


rolling_parts = []
for name in models:
    for split, fold, dates in rolling_blocks:
        part = fixed_lead(name, dates).assign(model=name, split=split, fold=fold)
        rolling_parts.append(part)
    print(f"Готова оценка {name}")

rolling_predictions = pd.concat(rolling_parts, ignore_index=True)
rolling_metrics = score(rolling_predictions, ["model", "months", "split"])
rolling_cv_metrics = score(rolling_predictions.query("split == 'valid'"), ["model", "months", "fold"])
display(rolling_metrics.sort_values(["split", "months", "model"]))
display(rolling_cv_metrics)

for name in models:
    client.log_text(run_ids[name], rolling_metrics.query("model == @name").to_csv(index=False),
                    "fixed_lead_metrics.csv")
    client.log_text(run_ids[name], rolling_cv_metrics.query("model == @name").to_csv(index=False),
                    "fixed_lead_cv_metrics.csv")

In [ ]:
cols = min(3, len(HORIZONS))
rows = (len(HORIZONS) + cols - 1) // cols

for name in models:
    for split in ("valid", "test"):
        block = rolling_predictions.query("model == @name and split == @split")
        start, end = block["target_dt"].min(), block["target_dt"].max()
        fig, axes = plt.subplots(
            rows, cols, figsize=(5 * cols, 2.8 * rows + 0.6), squeeze=False, layout="constrained"
        )
        fig.suptitle(f"{name} · {split} · {start:%Y-%m-%d} — {end:%Y-%m-%d}",
                     x=0.01, ha="left", fontsize=16)
        for h, ax in zip(HORIZONS, axes.flat):
            part = block[block["months"] == h].sort_values("target_dt")
            ax.set_title(f"{h} мес.", loc="left")
            draw_path(ax, part, split == "test" and SHOW_TEST_INTERVAL)
        for ax in axes.flat[len(HORIZONS):]:
            ax.set_visible(False)

        if cols > 1:
            fig.legend(*axes.flat[0].get_legend_handles_labels(), loc="outside upper right",
                       ncol=3, frameon=False)
        else:
            axes.flat[0].legend(frameon=False)
        fig.supylabel("OST_VAL")
        fig.supxlabel("Дата целевого остатка")
        plt.show()
        client.log_figure(run_ids[name], fig, f"{split}_fixed_lead_horizons.png")

## 7. Прогноз после последнего факта

Обе модели обучены на всей доступной истории до `last_fact`. В `future_predictions` есть каждый будущий шаг, его дата и 95% интервал.

In [ ]:
display(future_predictions[["model", "REP_DT", "target_dt", "forecast", "lower", "upper"]])

for name in models:
    rows = future_predictions[future_predictions["model"] == name]
    recent = y.dropna().tail(12 if monthly else 90)
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
    ax.plot(recent.index, recent, color="#185277", label="Последние факты")
    ax.plot(rows["target_dt"], rows["forecast"], color="#d45b26", label="Прогноз")
    ax.fill_between(rows["target_dt"], rows["lower"], rows["upper"], color="#d45b26",
                    alpha=0.15, label="95% интервал прогноза")
    ax.axvline(last_fact, color="0.5", linestyle=":")
    ax.set(title=f"{name} · выпуск {last_fact:%Y-%m-%d}", ylabel="OST_VAL", xlabel="Дата")
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)
    plt.show()
    client.log_figure(run_ids[name], fig, "future_forecast.png")

## 8. Коэффициенты моделей

После переобучения смотрим коэффициенты ARIMA и SARIMAX. На графике внешних признаков SARIMAX коэффициент показывает изменение в единицах `OST_VAL` на единицу признака при фиксированных остальных компонентах модели.

In [ ]:
tables = []
for name, model in models.items():
    table = model.params.rename("estimate").to_frame()
    table[["lower", "upper"]] = model.conf_int().to_numpy()
    table = table.rename_axis("term").reset_index().assign(model=name)
    tables.append(table)
    client.log_text(run_ids[name], table.to_csv(index=False), "coefficients.csv")

coefficients = pd.concat(tables, ignore_index=True)
display(coefficients)
sarimax_exog = (coefficients["model"] == "SARIMAX") & coefficients["term"].isin(exog.columns)
effects = coefficients[sarimax_exog].set_index("term")[["estimate", "lower", "upper"]]
effects = effects * target_scaling["SARIMAX"][1]
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.errorbar(effects["estimate"], effects.index, fmt="o",
            xerr=[effects["estimate"] - effects["lower"], effects["upper"] - effects["estimate"]])
ax.axvline(0, color="0.5", linewidth=1)
ax.set(title="Коэффициенты признаков SARIMAX", xlabel="Изменение OST_VAL на единицу признака")
client.log_figure(run_ids["SARIMAX"], fig, "coefficients.png")
plt.show()